<a href="https://colab.research.google.com/github/Deibid94/agente-bimbambuy/blob/main/ALURA_Tarea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Tarea de Alura

### **David Lopez Caceres**

In [2]:
!pip install -q langchain langchain-community langchain-groq faiss-cpu pypdf sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
from getpass import getpass
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Configuración de API Key (solicita la clave de forma segura)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Ingresa tu Groq API Key: ")

# 2. Carga y procesamiento del documento PDF
# Asegúrate de subir el archivo PDF a la sesión de Colab y poner la ruta correcta
pdf_path = "1a6f28a5-5804-40b1-9ce1-b9be2cb95881.pdf"

print("Cargando documento...")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 3. Fragmentación (Chunking) del texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)
splits = text_splitter.split_documents(docs)
print(f"Documento dividido en {len(splits)} fragmentos.")

# 4. Creación de Embeddings y Vectorstore (FAISS + HuggingFace local gratis)
print("Generando vectores e indexando en FAISS...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. Configuración del modelo de lenguaje (LLM)
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2
)

# 6. Definición del Prompt y la Cadena RAG
system_prompt = (
    "Eres un asistente virtual experto en responder preguntas operativas sobre documentos internos.\n"
    "Responde la pregunta basándote ÚNICAMENTE en el siguiente contexto extraído del documento.\n"
    "Si no sabes la respuesta o no está en el documento, di honestamente que no lo sabes.\n\n"
    "Contexto:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("¡Agente RAG inicializado con éxito!\n")


/tmp/ipykernel_1504/853663557.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ModuleNotFoundError: No module named 'langchain.chains'

In [4]:
# Función para realizar consultas
def preguntar_al_agente(pregunta):
    print(f" Pregunta: {pregunta}")
    response = rag_chain.invoke({"input": pregunta})
    print(f"🤖 Respuesta:\n{response['answer']}\n")
    print("-" * 60)

# Ejemplos de prueba sobre el documento de BimBam Buy
preguntar_al_agente("¿Cuáles son los motivos por los que BimBam Buy puede retener o revertir comisiones?")
preguntar_al_agente("¿Qué tono de comunicación debe tener un afiliado de la marca?")
preguntar_al_agente("¿Qué información se requiere para solicitar soporte?")


 Pregunta: ¿Cuáles son los motivos por los que BimBam Buy puede retener o revertir comisiones?


NameError: name 'rag_chain' is not defined

In [ ]:
agente-bimbambuy/
├── data/
│   └── documento.pdf         # Tu archivo PDF (Programa de Afiliados)
├── .env                      # Variables de entorno (API Keys)
├── requirements.txt          # Dependencias del proyecto
├── ingest.py                 # Script para procesar y crear la base vectorial
├── agent.py                  # Lógica del agente RAG
└── app.py                    # Interfaz CLI / Ejecución principal


langchain==0.2.14
langchain-community==0.2.12
langchain-groq==0.1.9
faiss-cpu==1.8.0.post1
pypdf==4.3.1
sentence-transformers==3.0.1
python-dotenv==1.0.1


In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

DATA_PATH = "data/documento.pdf"
DB_FAISS_PATH = "vectorstore/db_faiss"

def build_vectorstore():
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"No se encontró el archivo en {DATA_PATH}")

    print("📄 Cargando documento PDF...")
    loader = PyPDFLoader(DATA_PATH)
    docs = loader.load()

    print("✂️ Dividiendo texto en fragmentos...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )
    splits = text_splitter.split_documents(docs)

    print("🧠 Generando embeddings e indexando con FAISS...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(splits, embeddings)

    print("💾 Guardando base de datos vectorial localmente...")
    vectorstore.save_local(DB_FAISS_PATH)
    print("✅ Proceso de ingesta completado con éxito.")

if __name__ == "__main__":
    build_vectorstore()


In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

DB_FAISS_PATH = "vectorstore/db_faiss"

def get_rag_chain():
    if not os.path.exists(DB_FAISS_PATH):
        raise FileNotFoundError(
            "La base vectorial no existe. Ejecuta 'python ingest.py' primero."
        )

    # 1. Cargar la base vectorial persistida
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.load_local(
        DB_FAISS_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 2. Configurar el LLM
    llm = ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0.2,
        groq_api_key=os.getenv("GROQ_API_KEY")
    )

    # 3. Prompt RAG
    system_prompt = (
        "Eres un asistente virtual experto en responder preguntas operativas "
        "sobre políticas y procedimientos internos.\n"
        "Responde la pregunta basándote ÚNICAMENTE en el siguiente contexto.\n"
        "Si no sabes la respuesta o no aparece en el contexto, di textualmente "
        "que no dispones de esa información.\n\n"
        "Contexto:\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    # 4. Crear la cadena RAG
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    return create_retrieval_chain(retriever, question_answer_chain)


In [ ]:
from agent import get_rag_chain

def main():
    print("🤖 Inicializando Agente de IA para Documentos...")
    try:
        rag_chain = get_rag_chain()
    except Exception as e:
        print(f"❌ Error al iniciar el agente: {e}")
        return

    print(" Ready! Escribe 'salir' para finalizar.\n" + "-"*50)

    while True:
        query = input("\n👤 Pregunta: ")
        if query.lower() in ["salir", "exit", "quit"]:
            print("👋 ¡Hasta luego!")
            break

        if not query.strip():
            continue

        print("🔍 Buscando respuesta...")
        response = rag_chain.invoke({"input": query})
        print(f"\n🤖 Respuesta:\n{response['answer']}")
        print("-" * 50)

if __name__ == "__main__":
    main()


In [ ]:
# Entorno virtual
venv/
env/
.venv/

# Variables de entorno (API Keys)
.env

# Cache de Python
__pycache__/
*.pyc

# Base vectorial generada (se puede regenerar con ingest.py)
vectorstore/

# Archivos de SO/IDE
.DS_Store
.vscode/
.idea/


# Inicializar repositorio
git init

# Commit 1: Estructura inicial y configuración
git add .gitignore requirements.txt .env.example
git commit -m "chore: initial project structure and dependencies"

# Commit 2: Módulo de ingesta
git add ingest.py data/
git commit -m "feat: add document loader and vectorstore ingestion pipeline"

# Commit 3: Lógica RAG del Agente
git add agent.py
git commit -m "feat: implement RAG chain using LangChain and Groq"

# Commit 4: Interfaz CLI
git add app.py
git commit -m "feat: add CLI interface for user interactions"

# Commit 5: Documentación final
git add README.md
git commit -m "docs: add comprehensive README with architecture and execution guide"
